# 02 - Data Profiling and Data Quality Assessment

## Objective

This notebook evaluates the quality and analytical readiness of the London and New York Airbnb listings datasets before data cleaning and enrichment.

The assessment focuses on:

- Missing values and data completeness
- Duplicate records and listing ID uniqueness
- Price formatting and numeric validity
- Extreme price values and outlier identification
- Cross-city data-quality differences

## Input

- `data/raw/london/listings.csv.gz`
- `data/raw/new_york/listings.csv.gz`

## Output

- Missing-value reports for both cities
- Duplicate and primary-key validation results
- Numeric price fields for downstream analysis
- Outlier thresholds and flags

## Business Value

Reliable profiling prevents invalid or incomplete source data from affecting downstream analytics, star-schema modeling, machine learning segmentation, and dashboard insights.

In [1]:
import pandas as pd
import numpy as np

## 1. Dataset Loading

The London and New York listings datasets are loaded separately for quality assessment.

Keeping both cities separate at this stage enables city-level comparison of missingness, pricing structure, schema quality, and data availability before harmonization into a unified dataset.

In [2]:
london_listings = pd.read_csv("../data/raw/london/listings.csv.gz")

ny_listings = pd.read_csv("../data/raw/new_york/listings.csv.gz")

### Dataset Volume Observation

London contains 96,871 listings and New York contains 35,036 listings.

The difference in dataset size should be considered when comparing absolute listing counts. Cross-city comparisons should therefore use normalized measures, such as average price, median price, occupancy rate, or percentage-based metrics, rather than counts alone.

In [3]:
print("London:", london_listings.shape)

print("New York:", ny_listings.shape)

London: (96871, 79)
New York: (35036, 90)


## 2. Missing Value Analysis

Missing values are profiled to identify incomplete fields and assess their likely impact on later analytics.

The report captures:

- Missing-value count
- Missing-value percentage
- Fields with complete absence of data
- Fields with partial but potentially meaningful missingness

Missing values are not removed automatically. Later cleaning decisions are based on the business relevance of each field and whether the field is required for analysis, modeling, or reporting.

In [4]:
def missing_value_report(df):

    report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isnull().sum(),
        "missing_pct": round(df.isnull().mean() * 100, 2)
    })

    return report.sort_values(
        by="missing_pct",
        ascending=False
    )

In [5]:
london_missing = missing_value_report(
    london_listings
)

ny_missing = missing_value_report(
    ny_listings
)

In [6]:
london_missing.head(20)

,column,missing_count,missing_pct
license,license,96871,100.00
calendar_updated,calendar_updated,96871,100.00
neighbourhood_group_cleansed,neighbourhood_group_cleansed,96871,100.00
neighbourhood,neighbourhood,55662,57.46
neighborhood_overview,neighborhood_overview,55663,57.46
host_neighbourhood,host_neighbourhood,51021,52.67
host_about,host_about,47038,48.56
beds,beds,34920,36.05
price,price,34908,36.04
estimated_revenue_l365d,estimated_revenue_l365d,34908,36.04


In [7]:
ny_missing.head(20)

,column,missing_count,missing_pct
neighborhood_overview,neighborhood_overview,35036,100.00
host_since,host_since,35036,100.00
host_response_time,host_response_time,35036,100.00
host_thumbnail_url,host_thumbnail_url,35036,100.00
host_acceptance_rate,host_acceptance_rate,35036,100.00
host_response_rate,host_response_rate,35036,100.00
host_verifications,host_verifications,35036,100.00
neighbourhood,neighbourhood,35036,100.00
host_total_listings_count,host_total_listings_count,35036,100.00
host_neighbourhood,host_neighbourhood,35036,100.00


### Key Missingness Findings

London and New York show materially different completeness patterns.

London has substantial missingness in listing attributes such as beds, bathrooms, neighbourhood-related fields, host descriptions, and review scores. New York has particularly high missingness in beds and bathrooms, together with several host-related fields that are entirely unavailable in the source extract.

These findings support a cautious approach: fields that are unreliable or highly incomplete should not be treated as core drivers in the unified analytical model without explicit null handling or filtering.

## 3. Duplicate Record Analysis

Duplicate checks are performed at two levels:

1. Exact duplicate rows
2. Duplicate listing identifiers

Exact duplicate rows may indicate repeated ingestion or extraction problems. Duplicate listing IDs would be more serious because `id` is expected to uniquely identify an Airbnb listing and acts as the primary key for later joins.

In [8]:
print("London Duplicate Rows:",
      london_listings.duplicated().sum())

print("NY Duplicate Rows:",
      ny_listings.duplicated().sum())

London Duplicate Rows: 0
NY Duplicate Rows: 0


### Duplicate Validation Result

No exact duplicate rows and no duplicate listing IDs were found in either city dataset.

Therefore, the listing-level source datasets can be treated as having one record per listing for the purposes of cleaning, enrichment, and dimensional modeling.

In [10]:
print("London Duplicate Listing IDs:",
      london_listings["id"].duplicated().sum())

print("NY Duplicate Listing IDs:",
      ny_listings["id"].duplicated().sum())

London Duplicate Listing IDs: 0
NY Duplicate Listing IDs: 0


## 4. Price Format Validation

The source `price` field is stored as text and includes currency symbols and formatting characters.

Before performing statistical analysis, the field must be inspected and converted into a numeric representation. This enables price comparisons, descriptive statistics, outlier detection, and machine learning feature preparation.

In [11]:
print("London Price Sample:")
print(london_listings["price"].head(10))

print("\nNY Price Sample:")
print(ny_listings["price"].head(10))

London Price Sample:
0     $70.00
1    $149.00
2    $411.00
3        NaN
4    $210.00
5    $280.00
6     $90.00
7     $61.00
8    $340.00
9     $49.00
Name: price, dtype: str

NY Price Sample:
0    $117.27
1     $66.87
2     $77.17
3    $202.47
4    $365.97
5    $193.03
6    $104.35
7     $60.84
8        NaN
9        NaN
Name: price, dtype: str


## 4. Primary Key Validation

# Objective:
# Validate that listing identifiers are unique
# and suitable for relationship mapping.

In [ ]:
print("London Price Nulls:",
      london_listings["price"].isnull().sum())

print("NY Price Nulls:",
      ny_listings["price"].isnull().sum())

London Price Nulls: 34908
NY Price Nulls: 14343


## 5. Price Standardization for Profiling

A new field named `price_clean` is created by removing currency symbols and thousands separators, then converting the remaining value to a numeric data type.

The original `price` field is retained to preserve source traceability. The new numeric field is used only for analysis and later pipeline transformations.

In [ ]:
london_listings["price_clean"] = (
    london_listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
ny_listings["price_clean"] = (
    ny_listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
print(london_listings[["price", "price_clean"]].head(10))
print(ny_listings[["price", "price_clean"]].head(10))

print("\nLondon price_clean dtype:", london_listings["price_clean"].dtype)
print("NY price_clean dtype:", ny_listings["price_clean"].dtype)

     price  price_clean
0   $70.00         70.0
1  $149.00        149.0
2  $411.00        411.0
3      NaN          NaN
4  $210.00        210.0
5  $280.00        280.0
6   $90.00         90.0
7   $61.00         61.0
8  $340.00        340.0
9   $49.00         49.0
     price  price_clean
0  $117.27       117.27
1   $66.87        66.87
2   $77.17        77.17
3  $202.47       202.47
4  $365.97       365.97
5  $193.03       193.03
6  $104.35       104.35
7   $60.84        60.84
8      NaN          NaN
9      NaN          NaN

London price_clean dtype: float64
NY price_clean dtype: float64


### Price Conversion Result

The price conversion successfully creates a numeric `price_clean` field for both cities.

Missing source prices remain as null values rather than being replaced with artificial values. This preserves data integrity and prevents unsupported assumptions from influencing average-price or outlier calculations.

## 6. Domain Validation: Invalid Price Checks

A nightly Airbnb listing price should not be zero or negative.

This validation checks whether any converted price values violate that basic domain expectation. Records failing this rule would be flagged for investigation or excluded from price-based analytics.

In [ ]:
print("London zero/negative prices:",
      (london_listings["price_clean"] <= 0).sum())

print("NY zero/negative prices:",
      (ny_listings["price_clean"] <= 0).sum())

London zero/negative prices: 0
NY zero/negative prices: 0


### Domain Validation Result

No zero or negative cleaned prices were found in either London or New York.

This confirms that the non-null price values satisfy the basic domain rule required for later price analytics and modeling.

In [ ]:
print("London Price Summary")
print(london_listings["price_clean"].describe())

print("\nNY Price Summary")
print(ny_listings["price_clean"].describe())

London Price Summary
count    6.196300e+04
mean     2.299170e+02
std      4.437589e+03
min      7.000000e+00
25%      7.700000e+01
50%      1.350000e+02
75%      2.210000e+02
max      1.085147e+06
Name: price_clean, dtype: float64

NY Price Summary
count    20693.000000
mean       254.998857
std        440.478328
min          4.470000
25%         93.800000
50%        167.250000
75%        279.500000
max      15075.000000
Name: price_clean, dtype: float64


## 8. Extreme Price Record Inspection

The highest-priced listings are inspected manually to distinguish possible luxury-market records from likely source anomalies or data-entry issues.

This investigation is important because extreme prices can distort both statistical summaries and machine learning clustering models.

In [ ]:
london_listings.nlargest(
    10,
    "price_clean"
)[
    [
        "id",
        "name",
        "room_type",
        "property_type",
        "price_clean"
    ]
]

,id,name,room_type,property_type,price_clean
83405,1389784758661918909,Lux 2 Bed in Canary Wharf close to Excel & O2,Entire home/apt,Entire rental unit,1085147.0
10228,13841484,Bright & airy DoubleBed with EnSuite in Zone 2!,Entire home/apt,Entire rental unit,74100.0
77023,1311151886101957046,Walk To London Eye,Private room,Private room in rental unit,66189.0
54442,957005187369596707,Close To London Eye,Private room,Private room in rental unit,65000.0
57204,1009450636308513568,Close to London Eye (BOL),Private room,Private room in rental unit,58000.0
75312,1289793379859667497,Very Central Room - Walk to Eye,Private room,Private room in rental unit,58000.0
9489,13254774,No Longer Available,Private room,Private room in rental unit,53588.0
38591,558514003024408716,SHORT WALK TO LONDON EYE - DOUBLE ROOM (SUR),Private room,Private room in condo,50000.0
51162,904417963453083510,Twin Room Close to London Eye (RHI),Private room,Private room in rental unit,50000.0
63728,1124036005279942799,"Spacious, airy penthouse with stunning view",Entire home/apt,Entire rental unit,30812.0


In [ ]:
ny_listings.nlargest(
    10,
    "price_clean"
)[
    [
        "id",
        "name",
        "room_type",
        "property_type",
        "price_clean"
    ]
]

,id,name,room_type,property_type,price_clean
27603,1111666023012679487,Opulent Midtown Retreat with White-Gloved Service,Hotel room,Room in hotel,15075.00
27604,1111666150326773564,"Presidential Suite Park View King Bed, The Pierre",Hotel room,Room in hotel,15075.00
790,990529,Sunny Luxury Bedroom Guest Bathroom,Private room,Private room in rental unit,12279.07
914,1623431,Enormous Chrysler-View Bedroom/Bath,Private room,Private room in rental unit,12078.47
23189,830682282357157632,North Gallery Café,Entire home/apt,Tower,11061.80
23355,830656153550799267,"Eighth House, State of the Art NYC Venue",Entire home/apt,Tower,11061.80
16983,49920227,WELCOME HOME 15 MINUTES TO MANHATTAN BOOK TODAY,Private room,Private room in home,10944.83
13786,39553889,"Paper Factory, Executive King",Private room,Room in hotel,10000.00
13787,39554366,"Paper Factory, Superior Room w/ Double Beds",Private room,Room in hotel,10000.00
13788,39554839,"Paper Factory, Deluxe King",Private room,Room in hotel,10000.00


### Extreme Price Observation

Several listings contain exceptionally high prices relative to the wider market. Some records may represent genuine luxury inventory, while others appear implausible when compared with their room type or property description.

These records are retained in the original and master datasets for traceability. However, extreme values are later handled separately when preparing the machine learning clustering dataset so that they do not dominate segmentation results.

## 9. Percentile-Based Outlier Thresholds

Percentiles are used to define city-specific price thresholds.

Unlike a single fixed global threshold, percentile-based thresholds adapt to each city's market structure. The 99th percentile is used during profiling to flag unusually expensive listings for investigation.

In [ ]:
london_listings["price_clean"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25      77.0
0.50     135.0
0.75     221.0
0.90     360.0
0.95     500.0
0.99    1100.0
Name: price_clean, dtype: float64

In [ ]:
ny_listings["price_clean"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25      93.8000
0.50     167.2500
0.75     279.5000
0.90     476.5000
0.95     676.9140
0.99    1562.3508
Name: price_clean, dtype: float64

In [ ]:
print("London Listing  > £1100:", 
      (london_listings["price_clean"] > 1100).sum())

print("NY Listing  > $1562:", 
      (ny_listings["price_clean"] > 1562).sum())

London Listing  > £1100: 613
NY Listing  > $1562: 207


### Outlier Threshold Decision

The 99th percentile threshold is approximately £1,100 for London and $1,562 for New York.

Listings above these thresholds are not automatically deleted. They are flagged as potential outliers so that downstream analyses can decide whether to retain, exclude, or separately model them depending on the business purpose.

## Outlier Flagging

## 10. Outlier Flagging

A boolean outlier flag is created to preserve an explicit record of listings whose prices exceed the city-specific 99th percentile threshold.

Flagging rather than deleting protects auditability: analysts can still inspect high-value listings, while modeling workflows can choose to exclude extreme values when necessary.

In [ ]:
london_listings["price_outlier"] = (
    london_listings["price_clean"] > 1100
)

ny_listings["price_outlier"] = (
    ny_listings["price_clean"] > 1562
)

## Profiling Summary and Next Steps

The profiling process established the following:

1. Both city datasets have unique listing identifiers and contain no duplicate listing rows.
2. Missingness patterns differ significantly between London and New York, particularly for beds, bathrooms, and host-related fields.
3. Price is stored as text and requires conversion to a numeric field before analysis.
4. No non-null listing prices were zero or negative.
5. Both cities have highly right-skewed price distributions with extreme values that require explicit outlier handling.
6. Percentile-based thresholds provide a city-aware method for flagging unusually high prices.

The next notebook applies cleaning and standardization rules, including price cleaning, date conversion, missing-value decisions, outlier flagging, and feature creation.